### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="labour_inspection_compliance",
    dataset_year="2019",
    domain_str="industry & manufacturing",
    # Data Source
    dataset_source="Other",
    original_dataset_source_download_link="https://doi.org/10.18710/7U6TZP",
    download_description="""
Get the data from dataverse.

wget https://dataverse.no/api/access/datafile/:persistentId?persistentId=doi:10.18710/7U6TZP/HAV8AG &&  mv  ':persistentId?persistentId=doi:10.18710%2F7U6TZP%2FHAV8AG' data.csv
mkdir -p local-data-warehouse/labour_inspection_compliance && mv data.csv local-data-warehouse/labour_inspection_compliance
""",
    # References
    academic_reference_bibtex="""@inproceedings{flogard2022dataset,
  title={A dataset for efforts towards achieving the sustainable development goal of safe working environments},
  author={Flogard, Eirik Lund and Mengshoel, Ole Jakob},
  booktitle={Thirty-sixth Conference on Neural Information Processing Systems Datasets and Benchmarks Track},
  year={2022}
}
""",
    academic_reference_bibtex_key="flogard2022dataset",
    license="CC0 1.0",
    data_tags=["IID", "Spatial", "ForcedIIDFromTemporal"],
    curation_comments="""
We start with the data from dataverse. We create a task for the Non-compliance Classification Problem (NCP).

- We resolve the industry codes to their true names.
- We drop the checklist ID, as we use the checklist text as predictive signal.
- We remove two rows with missing values for IsRegisteredVATregister, as these seem to be data artifact given the data state.
- The data has spatial information (County).
- There are lot of undescribed numeric values. Thus, we do some general purpose preprocessing and drop all columns with less than 5 not nan values
- As it turns out, the data does not contain temporal information, although it is clearly a temporal task and has data from 2021 to 2019. It seems the curators have removed the temporal information. This can mean that there is not temporal leakage or relevance for the task, or that it was forgotten. In any case, we must treat the task as IID now.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="NonCompliance",
    problem_type="binary_classification",
    objective_metric_name="roc_auc",
    stratify_on="NonCompliance",
)

## Preprocessing

In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv(dataset_mold.path / "data.csv", sep=";")
print("Loaded data shape:", df.shape)


# Resolve industry codes
code_resolution = pd.read_csv("klass-version-30-codes.csv", sep=";")
code_as_number = pd.to_numeric(code_resolution["code"], errors="coerce")
int_codes_mask = code_as_number.notna() & (code_as_number % 1 == 0)
code_resolution = code_resolution[int_codes_mask].copy()
code_resolution["code"] = code_as_number[int_codes_mask].astype(int)
code_resolution = code_resolution.set_index(["code", "parentCode"])["name"].to_dict()
df["Industry Sub Area"] = df.apply(
    lambda row: code_resolution[(row["Industry Code"], row["Industry Main Area Code"])],
    axis=1
)
del code_resolution

sector_map = {
    "A": "Agriculture, forestry and fishing",
    "B": "Mining and quarrying",
    "C": "Manufacturing",
    "D": "Electricity, gas, steam and air conditioning supply",
    "E": "Water supply; sewerage, waste management and remediation activities",
    "F": "Construction",
    "G": "Wholesale and retail trade; repair of motor vehicles and motorcycles",
    "H": "Transportation and storage",
    "I": "Accommodation and food service activities",
    "J": "Information and communication",
    "K": "Financial and insurance activities",
    "L": "Real estate activities",
    "M": "Professional, scientific and technical activities",
    "N": "Administrative and support service activities",
    "O": "Public administration and defence; compulsory social security",
    "P": "Education",
    "Q": "Human health and social work activities",
    "R": "Arts, entertainment and recreation",
    "S": "Other service activities",
    "T": "Activities of household as employers; undifferentiated goods- and services-producing activities of households for own account",
    "U": "Activities of extraterritorial organisations and bodies",
}
df["Industry Main Area"] = df["Industry Main Area Code"].map(sector_map)

df = df[df["IsRegisteredVATregister"].notna()]

prev_cols = set(df.columns)
df = df.dropna(axis=1, thresh=5)
print("Columns dropped due to low non-nan count:", prev_cols - set(df.columns))

df = df.drop(columns=[
    'Checklist ID',
    "Industry Main Area Code",
])
as_cat_type = [
    "Industry Code",
    "IsRegisteredVATregister",
    "IsRegisteredEmploymentregister",
    "NonCompliance",
    "Currency code",
    "Fiscal accounting type",
]
as_string_type = [
    "Checklist Content",
    "Industry Main Area",
    "Industry Sub Area",
    "County",
]

for c in as_string_type:
    nan_mask = df[c].isna()
    df.loc[nan_mask, c] = np.nan
    df[c] = df[c].astype("string")

df[as_cat_type] = df[as_cat_type].astype("category")

# Drop duplicated columns
df = df.loc[:, ~df.T.duplicated()]

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

Loaded data shape: (63634, 581)
Columns dropped due to low non-nan count: {'Change in provision for non-drained risk', 'Exchange rate adjustment funds', 'Change in premiership', "Premium funds, defined contribution funds and pensioners' surplus funds", 'Portfolio result', 'Changes in the value of financial instruments valued at fair value', 'Effective share of gains and losses on hedging instruments in cash flow hedging', 'Retained earnings', 'Change in safety provision', 'Financial revenues', 'Compensations paid out', 'Transferred from funds for assessment differences', 'Change in technical provisions for the non-life insurance business', 'Cost of goods sold', 'Courses/subcurs courses', 'Share of other profit components using the equity method', 'Own bonds, certificates m.m.', 'Tax expense on extraordinary items', "The fund's securities portfolio", 'To (from) technical provisions for the non-life insurance business', 'Foreign securities at cost', 'Overdue, un paid pensions and ejacula

## Data Checks

In [3]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 63,634
Columns: 377

#### Duplicate Report
Total duplicate rows: 497 (0.78% of dataset)
Duplicate rows ignoring target: 676 (1.06% of dataset)
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [4]:
# Sample Rows
df_head

,Industry Code,IsRegisteredVATregister,IsRegisteredEmploymentregister,Business Age,County,Number of Employees,NonCompliance,Checklist Content,Currency code,Fiscal accounting type,Total receivables,Trade receivables,"Bank deposits, cash and the like",Company capital,Market-based bonds,Depreciation of fixed assets and intangible assets,Other operating cost,TOTAL ASSETS,Public fees owed,Total other long-term liabilities,Other market-based financial instruments,Machinery and plant,Total debt,Dividends,Group receivables,Other operating income,Total result,Transfer to/from funds,An increase in value for other financial instruments valued at fair value,Net income (loss) for the year,Goodwill,Loans to enterprises in the same group,Total funds consumed,Minority interests,Other profit components for IFRS enterprises,"Collected funds, gifts m.v.",Accrued dividends,Total purpose capital with self-imposed restrictions,Other operating expenses,Dividends on shares/dividends on primary capital certificates,Interest and similar income from loans to and receivables on customers,Net interest and credit commission income,Other fees and commission income,"Machinery, fixtures and means of transport",Issued by others,Leveling fund,Social costs,Total commission income and income from banking services,Interest and similar costs on issued securities,Interest and similar costs on deposits from and liabilities to customers,Write-down/reversal of write-down,Buildings and other permanent properties,Building loans,Total net loans and receivables from credit institutions,Assets taken over,"Total certificates, bonds and other interest-bearing securities with a fixed return",Total prepaid costs were not incurred and no income received,Total other operating expenses,Transferred to the savings bank's fund,Bond debt,Total liabilities to credit institutions,Deposits from and liabilities to customers without an agreed maturity,Pensions,Total commission costs and costs of banking,result of extraordinary records,Loss on guarantees m.v.,Transferred to gift funds and/or gifts,Total net change in value and gain/loss on foreign exchange and securities that are current assets,Interest and similar costs on responsible loan capital,Certificates and other short-term borrowing,Other belongings,Issued by the public,Annual results before minority interests,Transferred to other equity,Other interest-bearing securities,Other purpose capital with statutory restrictions,Total inventory,"Credit losses on certificates, bonds and other interest-bearing securities",Transferred to funds for assessment differences,Unspecified loss provisions,Fund for assessment differences,Debt letters that can be refinanced in central banks,Own non-amortized certificates,Obligations,Gift fund,Reinsurance obligations,Total prepaid costs and unearned income received,Financial assets that are measured at amortized cost,Financial assets that are measured at fair value,Sum incurred costs and did not receive earned revenue,Provision for unearned gross premium,"Daughter enterprises, affiliated enterprises and jointly controlled enterprises",Earned revenue from operational activities,Realized gain and loss on investments,Investments that are held to maturity,Other financial assets,Fund for changes in value,Net operating income from real estate,Total replacement costs at your own expense,– Reinsurance share of gross premiums earned,Total premium income at your own expense,Other insurance-related income,Other possessions denoted by their nature,Assets by tax,Intermediaries,Receivables in connection with reinsurance,Other prepaid costs and no income received,Sales costs,Ordinary result after tax expense,Group contributions made,Other current liabilities,Investments in equities and shares,Total retained earnings,Other financial instruments,Annual results by minority interests,Total long-term debt,Total intangible assets,Tax payable,"Total bank deposits, cash and the like",Investment in subsidiaries,Holdings of own 

In [5]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,Currency code,category,14003,22.01,9,"NOK, EUR, SEK, USD, CHF, DKK, USN, ISK, GBP"
1,Fiscal accounting type,category,14003,22.01,8,"STORE, BANK, IDEELL, SKADE, FRIV, FUNK, PENSJ, LIV"
2,Industry Code,category,0,0.00,84,"43, 41, 47, 56, 49, 88, 46, 10, 45, 84"
3,IsRegisteredVATregister,category,0,0.00,3,"Nei, Ja, Not specified"
4,IsRegisteredEmploymentregister,category,0,0.00,3,"Ja, Nei, Not specified"
5,NonCompliance,category,0,0.00,2,"1, 0"
6,Own non-amortized certificates,float64,63629,99.99,2,"0.0, 19000000.0"
7,Intermediaries,float64,63628,99.99,4,"76366000.0, 4859659.0, 5140099.0, 71000000.0"
8,Other assets,float64,63625,99.99,4,"2995000.0, 3176332.0, 240000.0, 11758000.0"
9,Receivables factoring,float64,63628,99.99,3,"0.0, 2297185815.0, 76190525.0"


In [6]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
Business Age,63634.0,2.011097e+02,1.976831e+02,-1.050000e+02,2.361000e+03
Number of Employees,63634.0,1.120812e+02,8.117329e+02,6.000000e+00,2.824900e+04
Total receivables,49051.0,1.770889e+08,2.123415e+09,-6.974970e+05,1.111610e+11
Trade receivables,47665.0,1.021949e+08,1.429322e+09,-1.138160e+06,8.741300e+10
"Bank deposits, cash and the like",46134.0,6.191823e+07,7.409474e+08,-3.855300e+07,4.812900e+10
Company capital,48713.0,6.494859e+07,8.818570e+08,-1.574047e+06,5.502100e+10
Market-based bonds,555.0,1.704854e+08,1.409295e+09,0.000000e+00,1.899600e+10
Depreciation of fixed assets and intangible assets,47206.0,2.002496e+07,1.315036e+08,-3.635000e+09,6.427000e+09
Other operating cost,49295.0,1.147271e+08,8.898446e+08,-6.028000e+09,5.226500e+10
TOTAL ASSETS,49631.0,2.541217e+09,6.117032e+10,-2.075746e+06,2.348272e+12


In [7]:
# Categorical Feature Statistics
cat_stats

value  \
column                         rank                                                                        
Checklist Content              1     Working agreements, HSE and working environment training for man...   
                               2     Working agreements, Working hour schedules, Building and equipme...   
                               3     HSE and working environment training for managers and worker rep...   
                               4     Working agreements, HSE and working environment training for man...   
                               5     HSE and working environment training for managers and worker rep...   
County                         1                                                                   Viken   
                               2                                                                    Oslo   
                               3                                                                Vestland   
                               4                                                               Trøndelag   
                               5                                                                Rogaland   
Currency code                  1                                                                     NOK   
                               2                                                                    <NA>   
                               3                                                                     EUR   
                               4                                                                     SEK   
                               5                                                                     USD   
Fiscal accounting type         1                                                                   STORE   
                               2                                                                    <NA>   
                               3                                                                    BANK   
                               4                                                                  IDEELL   
                               5                                                                   SKADE   
Industry Code                  1                                                                      43   
                               2                                                                      41   
                               3                                                                      47   
                               4                                                                      56   
                               5                                                                      49   
Industry Main Area             1                                                            Construction   
                               2     Wholesale and retail trade; repair of motor vehicles and motorcy...   
                               3                                                           Manufacturing   
                               4                               Accommodation and food service activities   
                               5                                 Human health and social work activities   
Industry Sub Area              1                                     Specialised construction activities   
                               2                                               Construction of buildings   
                               3                  Retail trade, except of motor vehicles and motorcycles   
                               4                                    Food and beverage service activities   
                               5                              Land transport and transport via pipelines   
IsRegisteredEmploymentregister 1                                                                      J

In [8]:
# Target Distribution
target_df

,count,pct
NonCompliance,,
1,47265,74.28
0,16369,25.72


## Task Curation

In [9]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended IID splits: n_repeats=3, n_splits=3, test_size=None


In [10]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
    ),
)

Using Stratified IID splits.


## Export

In [11]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
019c341f-2374-74a2-9859-42d23baa891e
c783e21a1fa0b2e354b8a54af7da525100f43ec06731c6a87b8e00d4daeee897
